> title : 프롬프트 생성 코드 <br>
> author : jacobgreen <br>

# 🔈 대회 소개
1. 대회명 : 건설공사 사고 예방 및 대응책 생성 <한솔데코 시즌3 AI 경진대회> <br>
2. 소개 : 시즌3 경진대회에서는 건설공사 사고 상황 데이터를 기반으로 사고 원인을 분석하고, 재발방지 대책 및 향후 조치 계획을 자동으로 생성하는 AI 모델을 개발하는 것이 목표입니다. 참가자들은 실제 사고 데이터를 학습하여 건설 현장의 사고 대응 역량을 향상시키고, 효과적인 안전관리 프로세스를 구축하는 AI 기술을 연구하게 됩니다. <br>
3. URL : https://dacon.io/competitions/official/236455/codeshare <br>
<img src="https://dacon.s3.ap-northeast-2.amazonaws.com/competition/236455/header_background.jpeg" style="width:100%; height:auto;">

In [1]:
! apt-get update -qq
! apt-get install fonts-nanum -qq > /dev/null

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import re
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
from tqdm.auto import tqdm
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 설치된 나눔고딕 폰트 경로 확인
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 2. Matplotlib 기본 폰트로 설정
fm.fontManager.addfont(font_path)  # 폰트 추가
mpl.rc("font", family="NanumGothic")  # 폰트 설정

# 3. 한글 폰트 캐시 삭제 및 다시 로드
mpl.font_manager._load_fontmanager()

pd.set_option("display.max_columns", None)  # 모든 컬럼 출력
pd.set_option("display.max_rows", 200)  # 모든 컬럼 출력
pd.set_option("display.max_colwidth", None)  # 컬럼 내용 생략 없음

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [39]:
import os
import sys

class SuppressStdoutStderr:
    def __enter__(self):
        self.stdout = sys.stdout
        self.stderr = sys.stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stderr.close()
        sys.stdout = self.stdout
        sys.stderr = self.stderr

In [45]:
%%time

import numpy as np
from sentence_transformers import SentenceTransformer

Embedding_Model = SentenceTransformer('jhgan/ko-sbert-sts', use_auth_token=False)

# 샘플에 대한 Cosine Similarity 산식
def cosine_similarity(Embedding_Model, gt, pred):

    pred_embed = Embedding_Model.encode(pred)
    gt_embed = Embedding_Model.encode(gt)

    dot_product = np.dot(gt_embed, pred_embed)
    norm_a = np.linalg.norm(gt_embed)
    norm_b = np.linalg.norm(pred_embed)
    return dot_product / (norm_a * norm_b) if norm_a != 0 and norm_b != 0 else 0

/usr/local/lib/python3.10/dist-packages/sentence_transformers/SentenceTransformer.py:195: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


CPU times: user 237 ms, sys: 126 ms, total: 363 ms
Wall time: 2.68 s


In [43]:
submission = pd.read_csv('/kaggle/input/project7/sample_submission.csv', encoding = 'utf-8-sig')

In [35]:
sub1 = pd.read_csv('/kaggle/input/project7/submission_0.6619778275489807.csv', encoding = 'utf-8-sig')
sub2 = pd.read_csv('/kaggle/input/project7/submission_0.6583671569824219.csv', encoding = 'utf-8-sig')
sub3 = pd.read_csv('/kaggle/input/project7/submission_0.6547933220863342.csv', encoding = 'utf-8-sig')
sub1 = sub1[['ID','재발방지대책 및 향후조치계획']]
sub2 = sub2[['재발방지대책 및 향후조치계획']]
sub3 = sub3[['재발방지대책 및 향후조치계획']]

In [36]:
data = pd.concat([sub1,sub2,sub3],axis=1)
data.columns = ['ID','sub1','sub2','sub3']
data.head()

,ID,sub1,sub2,sub3
0,TEST_000,펌프카 설치 시 지반 안정성 검토와 아웃리거 길이 조정 및 무게중심 균형 유지 지시를 통한 재발 방지 대책.,"펌프카 아웃트리거 지반 상태 사전 점검 및 안정성 확보, 작업 중 무게중심 조절 및 안전점검 강화, 작업자 안전교육 실시.","펌프카 아웃트리거 지반 상태 점검 및 보강, 무게중심 조절을 통한 안전성 확보, 작업 전 안전교육 실시 및 관리감독 강화."
1,TEST_001,"안전교육 강화와 보안면 착용 의무화, 숫돌 측면 사용 금지 및 작업 중 안전점검 강화를 통한 재발 방지 대책 및 향후 조치 계획.",안전교육 강화와 보안면 착용 의무화 및 숫돌 측면 사용 금지 지시.,안전장비 착용 의무화 및 작업자 교육 강화로 인한 재발 방지 대책 마련.
2,TEST_002,작업 전 안전교육 강화와 이동 시 주변 환경 주의 및 안전장구 착용을 통한 재발 방지 대책.,작업 전 안전교육 강화 및 작업장 위험요소 점검을 통한 재발 방지와 안전관리 교육 철저를 통한 향후 조치 계획.,작업 전 안전교육 강화 및 계단 이동 시 전방주시 교육 실시를 통한 재발 방지 대책 마련.
3,TEST_003,작업 후 현장 정리정돈 철저와 안전교육 강화 및 작업 전 안전점검 실시를 통한 재발 방지 대책.,작업발판 위 잔여자재 정리와 작업자 안전교육 강화를 통한 재발 방지 대책 및 향후 조치 계획.,작업발판 위 잔재물 정리와 안전교육 강화를 통한 재발 방지 대책 및 향후 조치 계획.
4,TEST_004,작업중 이동 시 안전교육 강화와 작업중 이동 시 안전점검 철저를 통한 재발 방지 대책.,작업 이동 시 안전교육 강화와 안전장구 착용을 통한 재발 방지 대책.,작업 전 안전교육 강화와 점심식사 이동 시 안전관리 철저를 통한 재발 방지 대책 및 향후 조치 계획.


In [41]:
%%time 

# 가중치 설정 (예시)
weights = np.array([0.35, 0.33, 0.32])

# 각 행마다 앙상블 수행
def ensemble_row(row, weights, model):
    with SuppressStdoutStderr():
        sentences = row[['sub1', 'sub2', 'sub3']].values
        embeddings = np.vstack([model.encode(s) for s in sentences])
        weighted_embedding = np.average(embeddings, axis=0, weights=weights)
        similarities = cosine_similarity([weighted_embedding], embeddings)[0]
        final_sentence = sentences[np.argmax(similarities)]
    return final_sentence

# 앙상블 수행 및 결과 컬럼 추가
data['ensemble'] = data.apply(lambda row: ensemble_row(row, weights, model), axis=1)
data.to_excel('./data.xlsx',index=False)

# 결과 확인
data[['ID', 'ensemble']].head()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

CPU times: user 9min 8s, sys: 9.93 s, total: 9min 18s
Wall time: 4min 57s


,ID,ensemble
0,TEST_000,"펌프카 아웃트리거 지반 상태 사전 점검 및 안정성 확보, 작업 중 무게중심 조절 및 안전점검 강화, 작업자 안전교육 실시."
1,TEST_001,"안전교육 강화와 보안면 착용 의무화, 숫돌 측면 사용 금지 및 작업 중 안전점검 강화를 통한 재발 방지 대책 및 향후 조치 계획."
2,TEST_002,작업 전 안전교육 강화와 이동 시 주변 환경 주의 및 안전장구 착용을 통한 재발 방지 대책.
3,TEST_003,작업발판 위 잔재물 정리와 안전교육 강화를 통한 재발 방지 대책 및 향후 조치 계획.
4,TEST_004,작업중 이동 시 안전교육 강화와 작업중 이동 시 안전점검 철저를 통한 재발 방지 대책.


In [44]:
data[['ID', 'ensemble']].head()

,ID,ensemble
0,TEST_000,"펌프카 아웃트리거 지반 상태 사전 점검 및 안정성 확보, 작업 중 무게중심 조절 및 안전점검 강화, 작업자 안전교육 실시."
1,TEST_001,"안전교육 강화와 보안면 착용 의무화, 숫돌 측면 사용 금지 및 작업 중 안전점검 강화를 통한 재발 방지 대책 및 향후 조치 계획."
2,TEST_002,작업 전 안전교육 강화와 이동 시 주변 환경 주의 및 안전장구 착용을 통한 재발 방지 대책.
3,TEST_003,작업발판 위 잔재물 정리와 안전교육 강화를 통한 재발 방지 대책 및 향후 조치 계획.
4,TEST_004,작업중 이동 시 안전교육 강화와 작업중 이동 시 안전점검 철저를 통한 재발 방지 대책.


In [46]:
test_results = data['ensemble']

In [48]:
# 문장 리스트를 입력하여 임베딩 생성
pred_embeddings = Embedding_Model.encode(test_results)
print(pred_embeddings.shape)  # (샘플 개수, 768)

# 최종 결과 저장
submission.iloc[:,1] = test_results         # test_results
submission.iloc[:,2:] = pred_embeddings
fname = f"./submission_emsemble.csv"
print(fname)
submission.to_csv(fname, index=False, encoding='utf-8-sig')

# check
submission['재발방지대책 및 향후조치계획'].value_counts().reset_index()

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

(964, 768)
./submission_emsemble.csv


,재발방지대책 및 향후조치계획,count
0,작업 전 안전교육 강화 및 작업장 위험요소 점검을 통한 재발 방지와 안전관리 교육 철저를 통한 향후 조치 계획.,64
1,작업전 안전교육 강화 및 작업장 위험요소 점검을 통한 재발 방지와 안전관리 교육 철저를 통한 향후 조치 계획.,12
2,작업 전 안전교육 실시와 작업장 위험요소 점검을 통한 재발 방지와 안전관리 교육 철저를 통한 향후 조치 계획.,6
3,작업자 안전교육 강화와 작업 중 위험요소 사전 점검 및 제거를 통한 재발 방지 대책.,6
4,작업 전 안전교육 실시와 작업 중 안전관리 철저를 통한 재발 방지 대책 및 향후 조치 계획.,6
...,...,...
849,고소작업대 안전난간 설치 및 작업자 안전고리 철저한 체결 교육 실시와 현장 안전점검 강화를 통한 재발 방지 대책 및 향후 조치 계획.,1
850,작업자 안전교육 강화와 작업 중 안전장비 착용 및 비계 인양 시 안전조치 철저를 통한 재발 방지 대책 및 향후 조치 계획.,1
851,작업 후 톱날 정지 확인 및 안전장치 점검을 통한 재발 방지 대책 마련.,1
852,작업 전 위험정보 제공 및 안전교육 강화와 작업 중 안전관리 철저를 통한 재발 방지 대책.,1
